# HybridGaussianProductFactor

<a href="https://colab.research.google.com/github/borglab/gtsam/blob/develop/gtsam/hybrid/doc/HybridGaussianProductFactor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    %pip install --quiet gtsam-develop
except ImportError:
    pass  # Not running on Colab, do nothing

`HybridGaussianProductFactor` is a specialized `DecisionTree` used internally during hybrid elimination. It represents a collection of `GaussianFactorGraph`s, each associated with a specific assignment of a set of discrete variables, along with a scalar value accumulated for that assignment.

Specifically, it's a `DecisionTree<Key, GaussianFactorGraphValuePair>`, where `GaussianFactorGraphValuePair` is `std::pair<GaussianFactorGraph, double>`.

Its primary role is to hold the intermediate results when eliminating a continuous variable that has discrete variables in its separator/conditional parents. When eliminating such a variable, the resulting factor on the separator involves:
1.  A set of purely continuous factors (a `GaussianFactorGraph`).
2.  A scalar term (representing accumulated constants or log-likelihoods).
This pair depends on the assignment of the discrete separator variables. The `HybridGaussianProductFactor` stores these pairs in a decision tree structure indexed by the discrete keys.

While crucial internally for elimination algorithms like `EliminateHybrid`, direct user interaction with `HybridGaussianProductFactor` is less common compared to `HybridGaussianFactor` or `HybridGaussianConditional`. It can be thought of as the precursor structure from which a `HybridGaussianFactor` or `HybridGaussianConditional` is formed after elimination.

In [ ]:
import gtsam
import numpy as np

from gtsam import (
    HybridGaussianProductFactor, HybridGaussianFactor,
    GaussianFactorGraph, GaussianFactor, JacobianFactor,
    DecisionTreeFactor
)
from gtsam.symbol_shorthand import X, D

ImportError: cannot import name 'HybridGaussianProductFactor' from 'gtsam' (c:\Users\porte\miniconda3\envs\gtsam\Lib\site-packages\gtsam\__init__.py)

## Conceptual Usage (Internal)

Imagine eliminating variable `X(0)` from factors that depend on `X(0)` and a discrete variable `D(0)`. The result might be represented internally as a `HybridGaussianProductFactor` before being converted into a `HybridGaussianFactor` or `HybridGaussianConditional`.

Let's simulate this by creating factor graphs for different discrete modes.

In [ ]:
dk0 = (D(0), 2) # Binary discrete key

# --- Define GFG and scalar for mode D0=0 ---
gfg0 = GaussianFactorGraph()
# Example factor remaining after eliminating X0, involving X1
gfg0.add(JacobianFactor(X(1), np.eye(1)*2, np.array([1.0]), gtsam.noiseModel.Unit.Create(1)))
scalar0 = 1.5 # Example scalar accumulated during elimination

# --- Define GFG and scalar for mode D0=1 ---
gfg1 = GaussianFactorGraph()
# Example factor remaining after eliminating X0, involving X1 and X2
gfg1.add(JacobianFactor(X(1), np.eye(1)*3, np.array([2.0]), gtsam.noiseModel.Unit.Create(1)))
gfg1.add(JacobianFactor(X(1), -np.eye(1), X(2), np.eye(1)*0.5, np.zeros(1), gtsam.noiseModel.Unit.Create(1)))
scalar1 = 2.8 # Example scalar

# --- Construct the HybridGaussianProductFactor ---
# This typically happens internally. We simulate it here.
# We need a DecisionTree structure mapping discrete assignments to (GFG, scalar) pairs.
# Assuming a constructor or method exists to build this (details may vary in bindings):

try:
    # Method 1: Using apply/map (conceptual)
    # leaves = {0: (gfg0, scalar0), 1: (gfg1, scalar1)}
    # product_factor = gtsam.HybridGaussianProductFactor(dk0, leaves) # Pseudo-code

    # Method 2: Building the DecisionTree explicitly (if possible)
    tree = gtsam.DecisionTreeGaussianFactorGraphValuePair() # Assumed class
    tree.add([dk0], [(gfg0, scalar0), (gfg1, scalar1)]) # Value order matters
    product_factor = gtsam.HybridGaussianProductFactor(tree)

    print("Simulated HybridGaussianProductFactor:")
    product_factor.print()

except Exception as e:
    print(f"Direct construction of HybridGaussianProductFactor may be complex/internal.")
    print(f"Error: {e}")

# --- Conversion (Conceptual) ---
# Internally, this product_factor might be converted to a HybridGaussianFactor
# by combining the graph keys and storing the (GFG, scalar) pairs.
# hybrid_gaussian_factor = gtsam.HybridGaussianFactor(dk0, product_factor) # Conceptual

## Operations

Since it's a specialized `DecisionTree`, it supports standard decision tree operations like `apply`, `combine`, etc., adapted for `GaussianFactorGraphValuePair` leaves. It also overloads `+` and `+=` operators for adding `GaussianFactor`s or `HybridGaussianFactor`s, which effectively adds the factors to the `GaussianFactorGraph` component of the appropriate leaves in the tree.

In [ ]:
# Create a simple HybridGaussianProductFactor P( | D0) initially empty for D0=0, D0=1
dk0 = (D(0), 2)
empty_pair = (gtsam.GaussianFactorGraph(), 0.0)
tree_base = gtsam.DecisionTreeGaussianFactorGraphValuePair()
tree_base.add([dk0], [empty_pair, empty_pair])
product_factor = gtsam.HybridGaussianProductFactor(tree_base)
print("Initial Empty HybridGaussianProductFactor:")
product_factor.print()

# --- Add a GaussianFactor ---
# This factor will be added to *all* GFG leaves
gf_shared = JacobianFactor(X(5), np.eye(1), np.zeros(1), gtsam.noiseModel.Unit.Create(1))
product_factor_p1 = product_factor + gf_shared # Uses operator+
print("\nAfter adding shared GaussianFactor:")
product_factor_p1.print()

# --- Add a HybridGaussianFactor ---
# This adds the corresponding Gaussian component to the GFG at each leaf
dk_h = (D(0), 2) # Must match the tree's discrete key(s)
gf_h0 = JacobianFactor(X(6), np.eye(1)*2, np.ones(1), gtsam.noiseModel.Unit.Create(1))
gf_h1 = JacobianFactor(X(7), np.eye(1)*3, np.ones(1)*2, gtsam.noiseModel.Unit.Create(1))
scalar_h0 = 0.1
scalar_h1 = 0.2
hybrid_gf = HybridGaussianFactor(dk_h, [(gf_h0, scalar_h0), (gf_h1, scalar_h1)])

product_factor_p2 = product_factor_p1 + hybrid_gf # Uses operator+
# Expected result:
# Leaf D0=0: GFG contains gf_shared, gf_h0. Scalar = scalar_h0
# Leaf D0=1: GFG contains gf_shared, gf_h1. Scalar = scalar_h1
print("\nAfter adding HybridGaussianFactor:")
product_factor_p2.print()

# --- += operator ---
product_factor_p1 += hybrid_gf # In-place version
print("\nOriginal updated with += HybridGaussianFactor:")
product_factor_p1.print() # Should now match product_factor_p2